# NekoClaw Kaggle Training — 2×T4 DDP
Author: Vaibhav — AGPLv3 + Commercial
Architecture: HalfKAv2_hm 8 buckets L1=1024 SCReLU hybrid i16/i8

Phone + face verification already done (free plan). This notebook trains on 2×T4 with DDP, AMP, lossless pause/resume.


In [ ]:
!nvidia-smi
!python -c "import torch; print(torch.cuda.device_count(), torch.cuda.get_device_name(0))"

In [ ]:
# Install deps
!pip install -q torch numpy pyyaml tqdm zstandard python-chess requests
!ls /kaggle/input/nekoclaw-data/ | head -20

In [ ]:
# Clone repo (or upload as dataset)
!git clone https://github.com/yourname/nekoclaw.git || true
%cd nekoclaw
!ls trainer/configs/

In [ ]:
# Resume if checkpoint exists in input (for Kaggle preemption)
!ls -lh /kaggle/input/nekoclaw-ckpt/ 2>&1 | head -20
import pathlib, os
ckpt = None
for p in ["/kaggle/input/nekoclaw-ckpt/last.pt", "checkpoints/last.pt"]:
    if pathlib.Path(p).exists():
        ckpt = p
        break
print("resume ckpt:", ckpt)

In [ ]:
# Launch DDP training on 2×T4
import os
os.environ["OMP_NUM_THREADS"]="4"
!torchrun --nproc_per_node=2 -m nekoclaw_trainer.train --config trainer/configs/kaggle.yaml --resume {ckpt or ''} 2>&1 | tee /kaggle/working/train.log


In [ ]:
# Export .nnue at end (or on interrupt)
!python -m nekoclaw_trainer.export --ckpt /kaggle/working/checkpoints/last.pt --out /kaggle/working/nekoclaw-1024x8-scReLU.nnue
!ls -lh /kaggle/working/*.nnue
# Test export parity: quick eval check
!python -c "from nekoclaw_trainer.model import NekoClawNet; import torch; m=NekoClawNet(); print('params', sum(p.numel() for p in m.parameters()))"

In [ ]:
# Handle Kaggle preemption: save last.pt to output
import signal, pathlib
print("If interrupted, last.pt is at checkpoints/last.pt and will be saved as output dataset")
!cp checkpoints/last.pt /kaggle/working/last.pt 2>&1 | head

### Local CPU training (your Debian laptop)
```bash
python -m nekoclaw_trainer.train --config trainer/configs/default.yaml --resume checkpoints/last.pt
# Pause: p or SIGUSR1 or Ctrl+C (graceful)
# Resume lossless: same command with --resume
```
### Convert data
```bash
python scripts/download_gm.py --source elite --elite-path /path/to/lichess_elite_db.pgn.zst --out data/raw/
python scripts/annotate.py --in data/raw/elite.pgn --out data/labeled.epd --engine stockfish --engine-path /usr/games/stockfish --depth 22 --threads 8 --hash 2048
python scripts/convert_pgn_to_bin.py --in data/labeled.epd --out data/train.bin --shard 2500000
```
